# Emotion Detection using CNN for MoodMate System

This notebook implements the **Emotion Detection module** of the MoodMate project.

A Convolutional Neural Network (CNN) is trained using the **FER2013 facial emotion dataset** to classify facial expressions into different emotions.

The detected emotion can later be used to recommend music based on the user's mood.

## Project Overview

This notebook implements the **Emotion Detection module** of the MoodMate project.

A Convolutional Neural Network (CNN) is trained on the **FER2013 facial emotion dataset** to classify facial expressions into emotions.

The detected emotion can later be used by the MoodMate system to recommend music based on the user's mood.

In [ ]:
import pandas as pd

df = pd.read_csv("fer2013.csv")
print(df.head())

   emotion                                             pixels     Usage
0        0  70 80 82 72 58 58 60 63 54 58 60 48 89 115 121...  Training
1        0  151 150 147 155 148 133 111 140 170 174 182 15...  Training
2        2  231 212 156 164 174 138 161 173 182 200 106 38...  Training
3        4  24 32 36 30 32 23 19 20 30 41 21 22 32 34 21 1...  Training
4        6  4 0 0 0 0 0 0 0 0 0 0 0 3 15 23 28 48 50 58 84...  Training


## 1. Import Libraries

In [1]:
import tensorflow as tf
print(tf.__version__)

2.21.0


In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Dropout, Flatten, BatchNormalization
from tensorflow.keras.utils import to_categorical

from sklearn.model_selection import train_test_split

## Dataset Description

The model is trained using the **FER2013 Facial Emotion Recognition dataset**.

Dataset characteristics:

- 48 × 48 grayscale facial images
- Images stored as pixel strings
- Each image labeled with an emotion

Emotion classes used in this project:

- Angry (including Disgust)
- Fear
- Happy
- Sad
- Surprise
- Neutral

## 2. Load FER2013 Dataset

In [ ]:
data = pd.read_csv("fer2013.csv")
print(data.head())
print("Emotion classes:", data.emotion.unique())

   emotion                                             pixels     Usage
0        0  70 80 82 72 58 58 60 63 54 58 60 48 89 115 121...  Training
1        0  151 150 147 155 148 133 111 140 170 174 182 15...  Training
2        2  231 212 156 164 174 138 161 173 182 200 106 38...  Training
3        4  24 32 36 30 32 23 19 20 30 41 21 22 32 34 21 1...  Training
4        6  4 0 0 0 0 0 0 0 0 0 0 0 3 15 23 28 48 50 58 84...  Training
Emotion classes: [0 2 4 6 3 5 1]


In [4]:
filename = "fer2013.csv"
df = pd.read_csv(filename)

print("Original emotion labels:", df['emotion'].unique())

Original emotion labels: [0 2 4 6 3 5 1]


In [7]:
data = pd.read_csv("fer2013.csv")

## 3. Merge Disgust into Angry

In [5]:
df['emotion'] = df['emotion'].replace(1,0)
df.loc[df['emotion'] > 1, 'emotion'] -= 1

In [8]:
data['emotion'] = data['emotion'].replace(1,0)
data.loc[data['emotion'] > 1, 'emotion'] -= 1

In [9]:
# Merge disgust (1) into angry (0)
data['emotion'] = data['emotion'].replace(1,0)

# Shift labels
data.loc[data['emotion'] > 1, 'emotion'] -= 1

print("New labels:", sorted(data.emotion.unique()))

New labels: [0, 1, 2, 3, 4]


## 4. Convert Pixel Strings to Images

In [10]:
pixels = data['pixels'].tolist()

X = []
for pixel in pixels:
    img = np.array(pixel.split(), dtype='float32')
    img = img.reshape(48,48)
    X.append(img)

X = np.array(X)
X = X / 255.0
X = np.expand_dims(X, -1)

y = data['emotion'].values

print("Dataset shape:", X.shape)

Dataset shape: (35887, 48, 48, 1)


## 5. Train Test Split

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=42
)

num_classes = 6

y_train = to_categorical(y_train, num_classes)
y_test = to_categorical(y_test, num_classes)

## 6. Build CNN Model using Keras

In [12]:
model = Sequential()

model.add(Conv2D(64,(3,3),activation='relu',input_shape=(48,48,1)))
model.add(BatchNormalization())
model.add(MaxPooling2D(2,2))

model.add(Conv2D(128,(3,3),activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(2,2))

model.add(Conv2D(256,(3,3),activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(2,2))

model.add(Flatten())

model.add(Dense(256,activation='relu'))
model.add(Dropout(0.5))

model.add(Dense(6,activation='softmax'))

model.summary()

C:\Users\afsia\AppData\Roaming\Python\Python310\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 46, 46, 64)     │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 46, 46, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 23, 23, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 21, 21, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 21, 21, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 10, 10, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 8, 8, 256)      │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 8, 8, 256)      │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 4, 4, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     1,048,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │         1,542 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,421,830 (5.42 MB)

 Trainable params: 1,420,934 (5.42 MB)

 Non-trainable params: 896 (3.50 KB)

## 7. Compile Model

In [13]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

## 8. Train Model

In [14]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test,y_test),
    epochs=30,
    batch_size=64
)

Epoch 1/30
505/505 ━━━━━━━━━━━━━━━━━━━━ 200s 386ms/step - accuracy: 0.4068 - loss: 1.4468 - val_accuracy: 0.4664 - val_loss: 1.2832
Epoch 2/30
505/505 ━━━━━━━━━━━━━━━━━━━━ 203s 402ms/step - accuracy: 0.5066 - loss: 1.1633 - val_accuracy: 0.4731 - val_loss: 1.2256
Epoch 3/30
505/505 ━━━━━━━━━━━━━━━━━━━━ 205s 407ms/step - accuracy: 0.5523 - loss: 1.0727 - val_accuracy: 0.5743 - val_loss: 1.0813
Epoch 4/30
505/505 ━━━━━━━━━━━━━━━━━━━━ 200s 397ms/step - accuracy: 0.5794 - loss: 1.0141 - val_accuracy: 0.5712 - val_loss: 1.0133
Epoch 5/30
505/505 ━━━━━━━━━━━━━━━━━━━━ 200s 396ms/step - accuracy: 0.6043 - loss: 0.9535 - val_accuracy: 0.5854 - val_loss: 1.0345
Epoch 6/30
505/505 ━━━━━━━━━━━━━━━━━━━━ 196s 389ms/step - accuracy: 0.6290 - loss: 0.9028 - val_accuracy: 0.5823 - val_loss: 1.0147
Epoch 7/30
505/505 ━━━━━━━━━━━━━━━━━━━━ 202s 399ms/step - accuracy: 0.6557 - loss: 0.8431 - val_accuracy: 0.5754 - val_loss: 1.0343
Epoch 8/30
505/505 ━━━━━━━━━━━━━━━━━━━━ 197s 389ms/step - accuracy: 0.6791 -

## 9. Evaluate Model

In [15]:
loss, acc = model.evaluate(X_test, y_test)
print("Test Accuracy:", acc)

113/113 ━━━━━━━━━━━━━━━━━━━━ 4s 33ms/step - accuracy: 0.6194 - loss: 1.9018
Test Accuracy: 0.6193925738334656


## 10. Save Model in Keras Format

In [16]:
model.save("emotion_cnn_model.keras")
print("Model saved as emotion_cnn_model.keras")

Model saved as emotion_cnn_model.keras


## Conclusion

In this notebook, a Convolutional Neural Network (CNN) was trained using the FER2013 dataset to detect facial emotions.

The trained model can classify facial expressions into emotions and serves as the **Emotion Detection module of the MoodMate system**.

This model can later be integrated with a music recommendation system to suggest songs based on the detected mood.